<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Duckiedrone IMU Architecture

Every IMU message a ROS 2 node receives starts on a chip a few millimeters across, soldered to the flight-controller board. On its way to the node it passes through sensor processing, an attitude estimator, a link between two computers, and a ROS 2 bridge, and each of them changes the data. This notebook follows that path, one hop at a time. The sequence is the same on virtual and physical Duckiedrones, except that the flight controller and sensors are simulated on a virtual one.

By the end of the notebook you should be able to answer:

- Which pieces of software touch the IMU data between the sensor chip and a subscriber?

- What does each one add to, or remove from, the signal?

- Where in the chain is the data still raw, and where is it already filtered?

## The IMU on the Duckiedrone

The IMU on the Duckiedrone is embedded in the flight-controller board rather than being a stand-alone Raspberry Pi sensor. This means the IMU chip is accessed by PX4 over the __Serial Peripheral Interface (SPI)__ bus. PX4 reads and processes its data at a high, board- and configuration-dependent rate, while the PX4 [Micro Air Vehicle Link (MAVLink)](https://mavlink.io/) stream used here is configured for $50\,\mathrm{Hz}$ (see [Table 1](#table-1)).

The flight controller needs these measurements at a high rate to keep the Duckiedrone stable. Once it has used the measurements, it sends them on for nodes to use:

1. The PX4 autopilot software reads the IMU data, applies its configured sensor calibrations,
   and runs its state estimator, the EKF2, which fuses the readings into a Duckiedrone attitude estimate.

2. PX4 sends the calibrated IMU measurements and fused attitude over MAVLink.

3. On the companion computer (the Raspberry Pi), the MAVROS2 ROS 2 bridge node receives the
   MAVLink messages and re-publishes them as `sensor_msgs/Imu` on the topic `/mavros/imu/data`.
   Some setups put the Duckiedrone under a namespace, which makes it
   `/DUCKIEDRONE_NAME/mavros/imu/data` instead, where `DUCKIEDRONE_NAME` is the name of your Duckiedrone.

Two things follow from this: the orientation that reaches ROS 2 is a fused estimate rather than a raw reading, and the calibration work, performed at the time of [flight controller configuration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/50-sensor-calibrations.html), has already been applied.

## The big picture

[Figure 1](#figure-1) traces IMU data through PX4 and MAVROS2 to the subscriber.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; max-width:100%; overflow-x:auto; text-align:left;">
             SPI                         MAVLink             ROS 2
+----------+     +---------------------+  50 Hz  +---------+       +-----------------+
| IMU chip |---->| PX4                 |-------->| MAVROS2 |------>| Subscriber node |
+----------+     | - Sensor processing |         +---------+       +-----------------+
                 | - EKF2 estimator    |
                 | - Flight control    |
                 +---------------------+
                            |
                            v
                        +--------+
                        | Motors |
                        +--------+</pre>
  <figcaption>
    Figure 1: IMU data flow from the flight-controller board to a ROS 2 subscriber node.
  </figcaption>
</figure>

The `Subscriber node` box stands for any ROS 2 node that reads the topic, such as the IMU reader in [Notebook 7](./7-interfacing-with-imu.ipynb).

[Table 1](#table-1) separates values set in the Duckiedrone's parameter file from rates that depend on the PX4 board, firmware, and scheduler. Some are controlled by PX4 parameters, accessible through [QGroundControl](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/42-flight-controller-configuration.html#installing-qgroundcontrol), while others are part of the IMU driver, estimator, or MAVLink stream setup.

<a id="table-1"></a>

Table 1: IMU data-flow rates and configuration sources.

| Stage | Rate | Set by |
| --- | --- | --- |
| IMU chip read by PX4 | Board- and configuration-dependent | IMU-driver configuration |
| Maximum gyro data rate in the PX4 sensor pipeline | $800\,\mathrm{Hz}$ | `IMU_GYRO_RATEMAX` parameter |
| Integrated IMU data | $250\,\mathrm{Hz}$ | `IMU_INTEG_RATE` parameter |
| EKF2 attitude estimate | Input- and scheduler-dependent | PX4 estimator and sensor-pipeline configuration |
| Streamed over MAVLink to the Raspberry Pi | $50\,\mathrm{Hz}$ target | PX4 MAVLink stream configuration |

The $50\,\mathrm{Hz}$ stream is a configured output target, not a guarantee that every component in the chain updates at a single fixed rate. A physical Duckiedrone should measure about $50\,\mathrm{Hz}$ with `ros2 topic hz`. Timing on a virtual Duckiedrone can differ, so measure its actual rate with the same command. The last section shows where the relevant settings live.

The MAVLink arrow in [Figure 1](#figure-1) joins two computers: PX4 runs on the flight controller, and MAVROS2 runs on the Raspberry Pi. On a physical Duckiedrone, the flight controller is a microcontroller running PX4 on a real-time operating system ([NuttX](https://github.com/apache/nuttx)); on a virtual one, PX4 runs in simulation. MAVROS2 runs under Linux on the Raspberry Pi. A __Universal Serial Bus (USB)__ connection joins the two boards on a physical Duckiedrone, while a virtual Duckiedrone uses a __User Datagram Protocol (UDP)__ connection instead.

That split is what sets the rates. PX4 uses the IMU in its flight-control loop, which needs predictable scheduling on the flight controller. The MAVLink stream is a separate, lower-rate feed for the Raspberry Pi. The fast, low-level loop keeps the Duckiedrone stable; application logic on the Raspberry Pi can use the lower-rate stream for slower tasks such as navigation.

## From the IMU chip to PX4

The IMU chip sits on the flight-controller board, and PX4 reads it over the SPI bus at a high rate selected by the board and its configuration. Nothing on the Raspberry Pi is involved in this process: PX4 needs these measurements to stabilize the Duckiedrone.

Measurements received by PX4 from the IMU are in the sensor's own frame. The measurements can sit slightly off zero even when nothing is moving, and that offset drifts as the board warms up. This is the same bias that defeated dead reckoning in [Notebook 2](./2-imu-estimation-and-drift.ipynb#why-the-bias-is-fatal-to-dead-reckoning). The chip initially reports raw counts rather than $\mathrm{m/s^2}$ and $\mathrm{rad/s}$, and its axes follow the board layout.

PX4's sensor-processing pipeline applies the offsets, scale factors, and frame rotation configured for the flight controller before downstream components use the data. The resulting measurements are expressed in PX4's body frame, __forward-right-down (FRD)__ ($x$ forward, $y$ right, $z$ down). This is the body frame of [Notebook 3](./3-rotation-representations.ipynb#euler-angles-and-the-rotation-matrix) turned upside down about $x$: $x$ still points forward, but $y$ and $z$ point the other way. MAVROS2 converts back before the data reaches ROS 2. The exact calibration values depend on the flight controller and [its configuration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/42-flight-controller-configuration.html).

## The flight controller's onboard attitude estimator

[Notebook 2](./2-imu-estimation-and-drift.ipynb#why-the-imu-data-matters) introduced EKF2, the estimator that turns these measurements into an attitude: it corrects roll and pitch against the proper-acceleration measurement, while yaw comes only from the gyroscope. On the Duckiedrone, EKF2 runs inside PX4 on the flight controller, and its output is an orientation quaternion ([Notebook 4](./4-quaternions-and-rotation-representations.ipynb)) relative to the world frame.

PX4 sends the calibrated angular rates and accelerations alongside that estimate. Those are sensor measurements, not outputs of the estimator.

The Duckiedrone has no magnetometer or GPS, so nothing on board measures heading, and the yaw estimate drifts as Notebook 2 showed. The downward time-of-flight rangefinder constrains height, not yaw.

## From PX4 to MAVROS2 over MAVLink

This is the step where the data crosses the MAVLink arrow in [Figure 1](#figure-1). PX4 has finished with the data and now has to get it to the Raspberry Pi, which on the physical Duckiedrone means a USB cable between the two boards and, on the virtual Duckiedrone, a UDP socket standing in for it. Either way they speak MAVLink, a compact binary protocol built for exactly this kind of link. The PX4 stream configuration targets $50\,\mathrm{Hz}$ for the fused attitude and IMU readings; the observed rate can be lower when the source data or link cannot sustain it.

On the Raspberry Pi, a MAVROS2 node holds that connection open and translates the data, turning incoming MAVLink messages into ROS 2 messages. Its `imu` plugin fills in a `sensor_msgs/Imu` and publishes it on `/DUCKIEDRONE_NAME/mavros/imu/data`. The `/DUCKIEDRONE_NAME` prefix appears only when the Duckiedrone publishes under a namespace, so on some setups the topic is simply `/mavros/imu/data`.

[Table 2](#table-2) attributes each ROS 2 IMU message field to its source.

<table id="table-2" class="lx-table">
  <caption>Table 2: Sources of fields in the ROS 2 IMU message.</caption>
  <thead>
    <tr>
      <th>Field</th>
      <th>Where it comes from</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>orientation</code> (quaternion)</td>
      <td>PX4's fused attitude estimate, converted to ROS conventions</td>
    </tr>
    <tr>
      <td><code>angular_velocity</code></td>
      <td>Calibrated gyroscope measurement, frame-converted by MAVROS2</td>
    </tr>
    <tr>
      <td><code>linear_acceleration</code></td>
      <td>Calibrated accelerometer proper-acceleration measurement, frame-converted by MAVROS2</td>
    </tr>
    <tr>
      <td><code>*_covariance</code></td>
      <td>Uncertainty when provided; all zeros mean unknown, and <code>orientation_covariance[0] = -1</code> means no orientation estimate</td>
    </tr>
  </tbody>
</table>

Despite its name, `linear_acceleration` carries the accelerometer's proper acceleration ([Notebook 1](./1-imu-basics.ipynb#the-accelerometer)), so a level, stationary Duckiedrone reads about $\tilde{f}_z = +9.81\,\mathrm{m/s^2}$ on the $z$ axis of the __forward-left-up (FLU)__ frame, not zero.

MAVROS2 does one more important manipulation on this data: it adjusts the reference-frame convention. PX4 works in FRD, but ROS works in FLU ($x$ forward, $y$ left, $z$ up), the body frame of Notebook 3, so MAVROS2 flips the $y$ and $z$ axes as it converts the data from PX4. That flip leaves $x$ alone: roll rate and forward proper acceleration keep the sign PX4 gave them, while pitch rate, yaw rate, and the $y$ and $z$ proper-acceleration components all come out with inverted signs.

The orientation quaternion crosses a second boundary on top of that one, from the __north-east-down (NED)__ world frame ($x$ north, $y$ east, $z$ down) PX4 works in to the __east-north-up (ENU)__ frame ($x$ east, $y$ north, $z$ up) ROS expects. Like the world frame of Notebook 3, ENU has $z$ up. The NED-to-ENU conversion is fixed; its axes do not follow the yaw estimate. Without an absolute heading reference, the estimated body attitude can have an initial heading offset and drift relative to those fixed world axes.

Note also what the sensor rows of [Table 2](#table-2) imply. `angular_velocity` and `linear_acceleration` are measurements that have been calibrated, but `orientation` is the output of an algorithm (EKF2), so it carries the drift from [the attitude estimator](#the-flight-controllers-onboard-attitude-estimator) with it. To inspect a message without the fused orientation, use `/mavros/imu/data_raw`. Its acceleration and gyroscope fields come from the `HIGHRES_IMU` MAVLink message. These are not direct chip readings either: PX4 has already applied calibration before MAVROS2 frame-converts them.

## From MAVROS2 to the subscriber node

The subscriber node reads `/DUCKIEDRONE_NAME/mavros/imu/data`; [Notebook 7](./7-interfacing-with-imu.ipynb) implements that reader.

The `sensor_msgs/Imu` message already contains calibrated, scaled, and frame-converted accelerations and angular rates. Its orientation is the estimator's fused output. The reader unpacks those values, converts the orientation quaternion to angles, and stores the result for display; it does not repeat raw-sensor calibration.

The MAVLink stream rate sets how often the message arrives, and that is much lower than the IMU sampling rate. [Notebook 7](./7-interfacing-with-imu.ipynb) measures the actual topic rate with `ros2 topic hz` rather than assuming a nominal value.

## Why not a Duckietown Postal Service (DTPS) driver, like the other sensors?

Other sensors on the Duckiedrone, such as the time-of-flight rangefinder and camera, reach the agent code, the code that controls the Duckiedrone, through a [DTPS](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html) driver and a ROS 2 bridge. The IMU skips all of that and arrives straight from MAVROS2. There are three closely related reasons:

1. The flight controller already owns the chip. It is soldered to the flight-controller board and read by PX4 firmware, so no driver on the Raspberry Pi touches it. A DTPS driver would ask PX4 for the data and republish it, which overlaps with MAVROS2's role.

2. The fast loop has to stay on the microcontroller. PX4 corrects the Duckiedrone's attitude every few milliseconds, and a general-purpose operating system cannot guarantee the same scheduling. Routing the IMU to the Raspberry Pi to close that loop would add latency and uncertainty.

3. The attitude estimate is already fused before it crosses over. PX4 completes the estimation before the Raspberry Pi receives it, so MAVROS2 avoids duplicating the estimator and gives ROS 2 consumers one shared attitude estimate.

On the Duckiedrone, then, the flight controller is the IMU driver and MAVROS2 is the ROS 2 bridge. That is useful for troubleshooting: the chip and PX4 continue operating whether or not anything is subscribed, so a quiet `/DUCKIEDRONE_NAME/mavros/imu/data` topic points to MAVROS2 or the link to PX4 rather than directly to the sensor.

## Further reading

[Using the ECL EKF](https://docs.px4.io/main/en/advanced_config/tuning_the_ecl_ekf.html) explains the EKF2 estimator that this notebook treats as a black box.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()